# Классы и объекты

1. Зачем нужны классы
2. Основные понятия
3. Инструменты: `property`, `classmethod`, `staticmethod`
4. Абстракция
5. Инкапсуляция
6. Наследование
7. Полиморфизм
8. Что будет на следующем занятии

После этого переходим к рефакторингу пайплайна. Разберем датаклассы если останется время

# Зачем нужны классы

Объектно-ориентированное программирование (ООП) — парадигма, в которой программа состоит из объектов, которые взаимодействуют друг с другом через интерфейсы (методы). Объект объединяет данные и поведение (функции для работы с этими данными) и предоставляет интерфейс, через который с ним работают.

Юз-кейсов классов несколько, мой любимый пример - бэкенд на Flask.

In [40]:
from flask import Flask

flask_app = Flask(__name__)

@flask_app.route('/')
def hello_world():
    return 'Hello World!'

flask_app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [21/Nov/2026 17:59:27] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/Nov/2026 17:59:27] "GET /favicon.ico HTTP/1.1" 404 -


Кейс номер 2 - собрать данные вместе и сделать реализацию независимой от логики внутреннего устройства
Сейчас молекула в нашем пайплайне — это словарь. И в целом, эту ошибку мы с вами и разбирали на первой паре:

- нигде не записано, какие поля у молекулы есть, — это знает только тот, кто его заполнял;
- опечатку в ключе не поймают ни PyCharm, ни mypy: ошибка всплывёт только при запуске (или не всплывёт вовсе);
- поведение (например, правило Липински) живёт где-то отдельно от данных.

In [22]:
molecule = {"mol_id": "M001", "smiles": "CC(=O)Oc1ccccc1C(=O)O", "MW": 180.16, "LogP": 1.31}

print(molecule["MW"])
print(molecule.get("Mw"))  # опечатка в ключе: никто не предупредит, просто None

180.16
None


Класс решает все три проблемы: объявляет поля, связывает данные с поведением и задаёт интерфейс для работы с объектом.

# Основные понятия

* **класс** — чертёж, по которому создаются объекты;
* **экземпляр (объект)** — конкретный объект, созданный по классу;
* **атрибут** — данные объекта или класса;
* **метод** — функция, определённая внутри класса;
* **`self`** — ссылка на объект, у которого вызван метод;
* **`__init__`** — метод, который заполняет только что созданный объект.

In [41]:
class Compound:
    """Химическое соединение."""

    kind = "small molecule"  # атрибут класса: общий для всех экземпляров

    def __init__(self, name: str, smiles: str) -> None:
        self.name = name  # атрибуты экземпляра: свои у каждого объекта
        self.smiles = smiles

    def describe(self) -> str:
        return f"{self.name}: {self.smiles}"


aspirin = Compound("aspirin", "CC(=O)Oc1ccccc1C(=O)O")
caffeine = Compound("caffeine", "Cn1cnc2c1c(=O)n(C)c(=O)n2C")

print(aspirin.describe())
print(type(aspirin), isinstance(aspirin, Compound))
print(aspirin.__dict__)  # атрибуты экземпляра хранятся в словаре

aspirin: CC(=O)Oc1ccccc1C(=O)O
<class '__main__.Compound'> True
{'name': 'aspirin', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O'}


`self` — это просто объект, у которого вызвали метод. Эти два вызова делают одно и то же:

In [42]:
print(aspirin.describe())
print(Compound.describe(aspirin))

aspirin: CC(=O)Oc1ccccc1C(=O)O
aspirin: CC(=O)Oc1ccccc1C(=O)O


**Ловушка с атрибутом класса.** Присваивание через объект не меняет атрибут класса, а создаёт у объекта свой атрибут с тем же именем.

In [44]:
print(aspirin.kind)
aspirin.kind = "drug"  # появился атрибут экземпляра, атрибут класса не изменился
print(aspirin.kind, caffeine.kind, Compound.kind)

Compound.kind = "organic compound"  # меняем у класса: видят все, у кого нет своего атрибута
print(aspirin.kind, caffeine.kind)
print(aspirin.__dict__)

drug
drug organic compound organic compound
drug organic compound
{'name': 'aspirin', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'kind': 'drug'}


`print(aspirin)` выводит неинформативную строку. Это поправляют два магических метода:
* `__repr__` — представление для разработчика (отладка, вывод в списках); в идеале по этой строке можно воссоздать объект;
* `__str__` — представление для пользователя (`print`, f-строки).

In [45]:
print(aspirin)

In [48]:
class Compound:
    """Химическое соединение."""

    def __init__(self, name: str, smiles: str) -> None:
        self.name = name
        self.smiles = smiles

    def describe(self) -> str:
        return f"{self.name}: {self.smiles}"

    def __repr__(self) -> str:
        return f"Compound(name={self.name!r}, smiles={self.smiles!r})"

    def __str__(self) -> str:
        return f"{self.name} ({self.smiles})"


caffeine = Compound("caffeine", "Cn1cnc2c1c(=O)n(C)c(=O)n2C")
print(caffeine)  # __str__
print([caffeine])  # в списке используется __repr__
print(repr(caffeine))

# Инструменты: `property`, `classmethod`, `staticmethod`

* **`@property`** — атрибут, который вычисляется при обращении. Значение нельзя «забыть обновить»: оно всегда считается из актуальных данных.
* **`@classmethod`** — метод получает класс (`cls`), а не объект. Обычно используется как альтернативный конструктор.
* **`@staticmethod`** — метод не получает ни объект, ни класс. По сути, обычная функция, которая логически относится к классу.

In [50]:
from typing import Self

from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")  # не печатать ошибки разбора SMILES


class Molecule:
    """Молекула с дескрипторами."""

    MAX_MOLECULAR_WEIGHT = 500
    MAX_LOG_P = 5
    MAX_HBD = 5
    MAX_HBA = 10

    def __init__(self, mol_id: str, smiles: str, mol: Chem.Mol) -> None:
        self.mol_id = mol_id
        self.smiles = smiles
        self.mol = mol
        self.molecular_weight = Descriptors.MolWt(mol)
        self.log_p = Crippen.MolLogP(mol)
        self.hbd = rdMolDescriptors.CalcNumHBD(mol)
        self.hba = rdMolDescriptors.CalcNumHBA(mol)

    @property
    def ro5_pass(self) -> bool:
        """Правило пяти Липински: вычисляется из дескрипторов."""
        return (
            self.molecular_weight <= self.MAX_MOLECULAR_WEIGHT
            and self.log_p <= self.MAX_LOG_P
            and self.hbd <= self.MAX_HBD
            and self.hba <= self.MAX_HBA
        )

    @classmethod
    def from_smiles(cls, mol_id: str, smiles: str) -> Self | None:
        """Альтернативный конструктор: проверяет SMILES до создания объекта."""
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return cls(mol_id, smiles, mol)

    @staticmethod
    def is_valid_smiles(smiles: str) -> bool:
        """Не нужны ни объект, ни класс — просто функция про молекулы."""
        return Chem.MolFromSmiles(smiles) is not None

In [51]:
aspirin = Molecule.from_smiles("M001", "CC(=O)Oc1ccccc1C(=O)O")
print(aspirin.molecular_weight, aspirin.ro5_pass)  # property — без скобок

print(Molecule.from_smiles("BAD", "C1CC"))  # незамкнутое кольцо -> None
print(Molecule.is_valid_smiles("CCO"), Molecule.is_valid_smiles("C1CC"))

180.15899999999996 True
None
True False


| | Что получает первым аргументом | Когда использовать |
|---|---|---|
| обычный метод | объект (`self`) | работа с данными конкретного объекта |
| `@classmethod` | класс (`cls`) | альтернативные конструкторы |
| `@staticmethod` | ничего | вспомогательная функция, которая логически относится к классу |

# 3-4 кита ООП
### Абстракция, наследование, инкапсуляция, полиморфизм

#### Абстракция
**Абстракция** — выбор тех свойств объекта, которые важны для задачи, и отказ от всех остальных.

У настоящей молекулы есть структура, 3D-конформации, заряды, спектры, поставщик, цена, место на складе, путь синтеза… Какие из этих свойств попадут в класс, зависит от задачи:

| Задача | Что важно | Что не нужно |
|---|---|---|
| наш пайплайн | `mol_id`, SMILES, дескрипторы, отпечаток | конформации, цена, склад |
| склад реактивов | каталожный номер, название, остаток в мг, полка | дескрипторы, отпечаток |
| докинг | SMILES, 3D-конформации, частичные заряды | цена, склад, правило Липински |

Одна и та же молекула в трёх задачах — три разных класса. И это не упущение, а решение.

Как решить, что входит в класс:
* **Какую задачу решаем?**
* **Какие свойства нам нужны для решения?**

# Инкапсуляция

Два смысла:
* Скрыть сложное поведение от юзеров
* контроль доступа: пользователи класса не должны случайно сломать его состояние.

Соглашения об именах:
* `name` — публичный атрибут;
* `_name` — внутренний: для класса и его наследников
* `__name` — «приватный»: Python искажает имя (name mangling)

`property` с сеттером позволяет проверять значение при записи.

In [52]:
class Sample:
    """Образец соединения в растворе."""

    def __init__(self, compound: str, concentration_mm: float) -> None:
        self.compound = compound
        self.concentration_mm = concentration_mm  # проходит через сеттер

    def estimate(self):
        self._process_concentration()

    def _process_concentration(self):
        pass

    def __process_concentration_secure(self):
        pass

    @property
    def concentration_mm(self) -> float:
        return self._concentration_mm

    @concentration_mm.setter
    def concentration_mm(self, value: float) -> None:
        if value < 0:
            raise ValueError("Concentration cannot be negative")
        self._concentration_mm = value


sample = Sample("aspirin", 10)
sample.concentration_mm = 5
sample.__process_concentration_secure()
try:
    sample.concentration_mm = -1
except ValueError as error:
    print(error)

AttributeError: 'Sample' object has no attribute '__process_concentration_secure'

In [55]:
class Secret:
    def __init__(self) -> None:
        self.__key = 42


secret = Secret()
print(secret.__dict__)  # имя искажено: _Secret__key
print(secret._Secret__key)  # но добраться до него всё равно можно
secret._Secret__key = 43
print(secret._Secret__key)

{'_Secret__key': 42}
42
43


Python не запрещает доступ по-настоящему, как C++ или Java: «Мы все здесь взрослые люди» (Гвидо ван Россум).

# Наследование (кратко)

Можно строить иерархии классов. Класс-наследник получает атрибуты и методы родителя и может переопределять их. `super()` вызывает реализацию родителя.

In [56]:
class Drug(Compound):
    """Лекарство — соединение со статусом одобрения."""

    def __init__(self, name: str, smiles: str, approved: bool) -> None:
        super().__init__(name, smiles)
        self.approved = approved

    def describe(self) -> str:
        status = "approved" if self.approved else "investigational"
        return f"{super().describe()} [{status}]"


ibuprofen = Drug("ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O", approved=True)
print(ibuprofen.describe())
print(isinstance(ibuprofen, Compound), issubclass(Drug, Compound))

ibuprofen: CC(C)Cc1ccc(cc1)C(C)C(=O)O [approved]
True True


# Полиморфизм (кратко)

Возможность использовать разные данные, если у них определен подходящий интерфейс

In [57]:
for compound in [caffeine, ibuprofen]:
    print(compound.describe())  # у Drug своя реализация describe

caffeine: Cn1cnc2c1c(=O)n(C)c(=O)n2C
ibuprofen: CC(C)Cc1ccc(cc1)C(C)C(=O)O [approved]


**Утиная типизация**: «если что-то ходит как утка и крякает как утка, то это утка». Важно наличие нужного метода, а не родительский класс.

In [58]:
class Student:
    def __init__(self, name: str) -> None:
        self.name = name

    def describe(self) -> str:
        return f"Проблема очень сложная, но мы ee решим"


for item in [caffeine, ibuprofen]:
    print(item.describe())

# встроенные функции тоже полиморфны
print(len("CCO"), len([1, 2, 3]), len({"MW": 180.16}))

caffeine: Cn1cnc2c1c(=O)n(C)c(=O)n2C
ibuprofen: CC(C)Cc1ccc(cc1)C(C)C(=O)O [approved]
Проблема очень сложная, но мы ee решим
3 3 1


# Датаклассы

`@dataclass` генерирует за нас `__init__`, `__repr__` и `__eq__` по списку полей с аннотациями типов.

Полезные настройки:
* `frozen=True` — объект нельзя изменить после создания;
* `field(repr=False, compare=False)` — поле не выводится в `repr` и не участвует в сравнении;
* `ClassVar` — атрибут класса, а не поле: в `__init__` он не попадает.

In [35]:
from dataclasses import FrozenInstanceError, dataclass, field
from typing import ClassVar


@dataclass(frozen=True)
class Molecule:
    """Молекула с дескрипторами."""

    MAX_MOLECULAR_WEIGHT: ClassVar[float] = 500
    MAX_LOG_P: ClassVar[float] = 5
    MAX_HBD: ClassVar[int] = 5
    MAX_HBA: ClassVar[int] = 10

    mol_id: str
    smiles: str
    mol: Chem.Mol = field(repr=False, compare=False)
    molecular_weight: float
    log_p: float
    hbd: int
    hba: int

    @classmethod
    def from_smiles(cls, mol_id: str, smiles: str) -> Self | None:
        """Альтернативный конструктор: проверяет SMILES до создания объекта."""
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return cls(
            mol_id=mol_id,
            smiles=smiles,
            mol=mol,
            molecular_weight=Descriptors.MolWt(mol),
            log_p=Crippen.MolLogP(mol),
            hbd=rdMolDescriptors.CalcNumHBD(mol),
            hba=rdMolDescriptors.CalcNumHBA(mol),
        )

    @property
    def ro5_pass(self) -> bool:
        """Правило пяти Липински: вычисляется из дескрипторов."""
        return (
            self.molecular_weight <= self.MAX_MOLECULAR_WEIGHT
            and self.log_p <= self.MAX_LOG_P
            and self.hbd <= self.MAX_HBD
            and self.hba <= self.MAX_HBA
        )


aspirin = Molecule.from_smiles("M001", "CC(=O)Oc1ccccc1C(=O)O")
print(aspirin)  # __repr__ сгенерирован, объект RDKit не выводится

Molecule(mol_id='M001', smiles='CC(=O)Oc1ccccc1C(=O)O', molecular_weight=180.15899999999996, log_p=1.3101, hbd=1, hba=3)


`__eq__` сравнивает поля. Объект RDKit сравнивается по идентичности, поэтому без `compare=False` две молекулы из одного SMILES были бы «не равны».

In [36]:
aspirin_again = Molecule.from_smiles("M001", "CC(=O)Oc1ccccc1C(=O)O")
print(aspirin is aspirin_again)  # разные объекты
print(aspirin == aspirin_again)  # но равные данные

False
True


`frozen=True`: после создания объект не меняется.

In [37]:
try:
    aspirin.log_p = 0
except FrozenInstanceError as error:
    print(error)

cannot assign to field 'log_p'


`ClassVar`: порог общий для всех молекул. Меняем у класса — меняется результат у всех экземпляров.

In [38]:
print(aspirin.ro5_pass)
Molecule.MAX_LOG_P = 1
print(aspirin.ro5_pass)  # LogP аспирина 1.31 > 1
Molecule.MAX_LOG_P = 5  # возвращаем как было

True
False


Изменяемое значение по умолчанию (список, словарь) напрямую задать нельзя — та же ловушка, что с аргументами функций по умолчанию. Для них есть `default_factory`.

In [39]:
try:

    @dataclass
    class Experiment:
        name: str
        compounds: list[str] = []

except ValueError as error:
    print(error)


@dataclass
class Experiment:
    name: str
    compounds: list[str] = field(default_factory=list)  # новый список для каждого объекта


first = Experiment("screening")
second = Experiment("follow-up")
first.compounds.append("aspirin")
print(first, second)

mutable default <class 'list'> for field compounds is not allowed: use default_factory
Experiment(name='screening', compounds=['aspirin']) Experiment(name='follow-up', compounds=[])


Датаклассы бывают в двух ролях:
* данные плюс поведение — как `Molecule` с `ro5_pass` и `from_smiles`;
* только данные — например, настройки запуска. Такой класс увидим в пайплайне.

# Что будет на следующем занятии

* добавим загрузку молекул из других источников и построим для них иерархию классов;
* наследование, MRO и `super()` подробно;
* абстрактные классы и протоколы: как зафиксировать общий интерфейс;
* принципы SOLID;
* паттерны проектирования, в первую очередь Стратегия.

А теперь — к пайплайну.